# Healthcare Policy RAG Pipeline
**Flow:** Load PDFs → Chunk → Embed → Vector Store (Chroma) → Retrieve → Answer with LLM

## 1. Setup & Configuration

In [ ]:
# %pip install -q openai python-dotenv pandas langchain-community langchain-text-splitters langchain-openai langchain-chroma pypdf

import os
from dotenv import load_dotenv

load_dotenv("../.env", override=True)

AZURE_OPENAI_ENDPOINT        = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY         = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_MODEL           = os.getenv("AZURE_OPENAI_MODEL")
AZURE_OPENAI_EMBEDDING_MODEL = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL")
API_VERSION                  = "2024-02-01"

missing = [k for k, v in {
    "AZURE_OPENAI_ENDPOINT": AZURE_OPENAI_ENDPOINT,
    "AZURE_OPENAI_API_KEY": AZURE_OPENAI_API_KEY,
    "AZURE_OPENAI_MODEL": AZURE_OPENAI_MODEL,
}.items() if not v]
if missing:
    raise ValueError(f"Missing in ../.env: {', '.join(missing)}")

print("Endpoint configured :", bool(AZURE_OPENAI_ENDPOINT))
print("API key configured  :", bool(AZURE_OPENAI_API_KEY))
print("Chat model          :", AZURE_OPENAI_MODEL)
print("Embedding model     :", AZURE_OPENAI_EMBEDDING_MODEL)

## 2. Load Policy Documents

In [ ]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

POLICY_DIR = Path("notebook/healthcare_policies")

documents = []
for pdf in sorted(POLICY_DIR.glob("*.pdf")):
    documents.extend(PyPDFLoader(str(pdf)).load())

print(f"Loaded {len(documents)} page(s) from {len(list(POLICY_DIR.glob('*.pdf')))} PDF(s)")

## 3. Chunk Documents

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(documents)

print(f"Total chunks: {len(chunks)}")

## 4. Embeddings & Vector Store

In [ ]:
from langchain_openai import AzureOpenAIEmbeddings
from langchain_chroma import Chroma

embeddings = AzureOpenAIEmbeddings(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    model=AZURE_OPENAI_EMBEDDING_MODEL or "text-embedding-3-small",
    api_version=API_VERSION,
)

# Fresh collection each run — avoids duplicate chunks piling up on re-runs
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db",
    collection_name="healthcare_policies",
)

retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 5})
print("Vector store ready.")

## 5. Ask Questions (RAG)

In [ ]:
from langchain_openai import AzureChatOpenAI

llm = AzureChatOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    model=AZURE_OPENAI_MODEL,
    api_version=API_VERSION,
)

def ask(question: str, show_sources: bool = False) -> str:
    """Retrieve relevant chunks and answer using only that context."""
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)

    prompt = f"""Answer the question using only the provided context.

Context:
{context}

Question:
{question}"""

    answer = llm.invoke(prompt).content

    if show_sources:
        print("--- Retrieved sources ---")
        for doc in docs:
            src = Path(doc.metadata.get("source", "?")).name
            print(f"[{src} | p.{doc.metadata.get('page', '?')}] {doc.page_content[:120]}...")
        print("-" * 40)

    return answer

In [ ]:
print(ask("What is the prior authorization process?", show_sources=True))